#Extract

In [1]:
import os
import requests
import json
from datetime import datetime
from io import StringIO
import pandas as pd
from google.cloud import storage

In [2]:
API_KEY = "7c77e6d6a551915e3890b158"
url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/latest/USD"

In [3]:
response = requests.get(url)
data = response.json()

In [4]:
if response.status_code == 200 and data["result"] == "success":
  rates = data['conversion_rates']
  print("USD to GBP:", rates["GBP"])
else:
  print("API Error:", data)



USD to GBP: 0.7428


In [5]:
if response.status_code != 200 or data['result'] != "success":
  raise RuntimeError(f"API Error: {data}")

In [6]:
date = data['time_last_update_utc']
dt = datetime.strptime(date, "%a, %d %b %Y %H:%M:%S %z")
formatted_date = dt.strftime("%Y-%m-%d")
print(f"The date is {formatted_date}")

The date is 2026-01-13


In [7]:
dir = f"data/raw/api_currency/{formatted_date}"
os.makedirs(dir, exist_ok=True)

In [8]:
path = os.path.join(dir, "response.json")
with open(path, "w", encoding="utf-8") as f:
  json.dump(data, f, indent=2)

In [9]:
cs = pd.read_csv("data/raw/clickstream.csv", chunksize=50000)
tr = pd.read_csv("data/raw/transactions.csv", chunksize=50000)

#Transform/ Load


In [10]:
def transform_tr(df):
  df.columns = df.columns.str.lower().str.strip()
  df.rename(columns={"txn_id":"transaction_id", "txn_time":"transaction_time"}, inplace=True)
  df["transaction_time"] = pd.to_datetime(df["transaction_time"], utc=True)
  df = df.drop_duplicates(subset=["transaction_id"])
  df["amount_in_usd"] = df.apply( lambda row: round(row["amount"]/rates[row["currency"]],2), axis = 1)
  return df


In [11]:
def transform_cs(df):
  df.columns = df.columns.str.lower().str.strip()
  df["click_time"] = pd.to_datetime(df["click_time"], utc=True)
  df = df.drop_duplicates(subset=["session_id"])
  return df

In [12]:
def write_to_gcs(df, name, part):
  file_name = f"{name}part{str(part)}.csv"
  path = f"{name}/ingest_date={formatted_date}/{file_name}"

  csv_buffer = StringIO()
  df.to_csv(csv_buffer, index=False)

  client = storage.Client()
  bucket = client.bucket("week_1_bucke")
  blob = bucket.blob(path)
  blob.upload_from_string(csv_buffer.getvalue(), content_type="text/csv")
  print(f"Uploaded {df.shape[0]} rows to gs://week_1_bucke/{path}")


In [13]:
for i, chunk in enumerate(cs):
  transformed_cs = transform_cs(chunk)
  write_to_gcs(transformed_cs, "clickstream",i)

Uploaded 50000 rows to gs://week_1_bucke/clickstream/ingest_date=2026-01-13/clickstreampart0.csv
Uploaded 50000 rows to gs://week_1_bucke/clickstream/ingest_date=2026-01-13/clickstreampart1.csv
Uploaded 50000 rows to gs://week_1_bucke/clickstream/ingest_date=2026-01-13/clickstreampart2.csv
Uploaded 50000 rows to gs://week_1_bucke/clickstream/ingest_date=2026-01-13/clickstreampart3.csv


In [14]:
for i, chunk in enumerate(tr):
  transformed_tr = transform_tr(chunk)
  write_to_gcs(transformed_tr, "transactions",i)

Uploaded 50000 rows to gs://week_1_bucke/transactions/ingest_date=2026-01-13/transactionspart0.csv
Uploaded 50000 rows to gs://week_1_bucke/transactions/ingest_date=2026-01-13/transactionspart1.csv


In [15]:
print("done")

done
